LAUNCH LASER AND FILTER WHEEL - MAKE SURE LASER IS DISCONNECTED FROM iBEAM SMART SOFTWARE

In [10]:
%load_ext autoreload
%autoreload 2


import Components as comp
import socket
import sys

if socket.gethostname() == "ph-photonbec5":
    sys.path.append(r"D:/Control/PythonPackages/")

components = dict()

filter_wheel = comp.FilterWheel(allowed_filter_positions=[0, 1, 2, 3, 4, 5])
laser = comp.Toptica_Laser()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


START DATA ACQUISITION - IF YOU WANT TO QUIT DURING A SCAN YOU HAVE TO STOP SERVER

In [7]:
#POWER SCANS INCLUDED - Manually setting the exposition params.
%load_ext autoreload
%autoreload 2

import numpy as np
import Components as comp
import time
import socket
import sys
import os

if socket.gethostname() == "ph-photonbec5":
    sys.path.append("D:/Control/PythonPackages/")
    sys.path.append("D:/Control/PiezoController/")

    
from pbec_analysis import make_timestamp,ExperimentalDataSet, CameraData

def save_image(image,stage_position,exposure_time,pwr):
    timestamp = make_timestamp(precision=0)
    dataset = ExperimentalDataSet(timestamp)
    camera_data = CameraData(timestamp,extension='_' + str(round(stage_position,1)) + '_' + str(pwr) + '_' + str(int(exposure_time)) + '_' + str(np.amax(image)) + '_single' + '.png')
    camera_data.data = image
    dataset.dataset["CavityCamera" + str(stage_position) + str(pwr) + str(exposure_time) + str(np.amax(image))+ 'single'] = camera_data
    dataset.saveAllData()
    time.sleep(0.2)


min_exp = 100
max_exp = 1999994
standard_exp = 50000 
number_of_pics = 10 # Amount of pics to take at each position of the stage
n_run = 1 # Amount of run to perform for each power
#_________________________________________________________CHANGE HERE____________________________________________________________
position_list = np.append(np.append(np.arange(0,90,5),np.arange(90,300,10)),np.arange(300,630,30))
position_list = np.append(np.append(np.arange(0,90,5),np.arange(90,300,10)),np.arange(300,630,30))
power_list = np.append(np.append(np.arange(35,70,1)/1000,np.arange(70,100,5)/1000),np.arange(100,150,10)/1000) #CHANGE HERE
#power_list = np.array([25.2])/1000
pulsed = False
expos = 100

filter_positions = { # Preset the filter position for each power value - Need to be done manually, do it everytime before taking measurements
    30: 1,
    35: 2,
    40: 3,
    50: 4,
    # Add more power settings as needed
}
#_________________________________________________________________________________________________________________________________

with comp.Thor_Camera(min_exp, max_exp, measure=True, max_frames=1) as camera:
    with comp.Translation_Stage() as stage:
        camera.change_exposure(expos)
        for power in power_list:
            if pulsed:
                laser.set2(power)
                print('Scanning for pulsed operation')
            else:
                laser.set(power)
                print('Scanning for CW operation')
            # if power <=28/1000 and power >=27/1000:
            #     camera.change_exposure(1100)
            # else:
            #     camera.change_exposure(min_exp)
            print('Exposure is : ', camera.get_exposure())
            print('Measurement for power: ', power*1000)
            try:
                if power*1000 in filter_positions:
                    filter_position = filter_positions[power*1000]
                    filter_wheel.set(filter_position)  # Set the filter position accordingly
                    print(f'Filter position set to: {filter_position+1} for power {power*1000}')
                    
                for j in range(n_run):
                    exposure = camera.get_exposure()
                    print('Data Set #',j+1)
    
                    for k in range(len(position_list)):
                        stage.set(position_list[k])
                        position = stage.get_position()
                        print('Now at Position : ', position)
    
                        for k in range(number_of_pics):
                            img = camera.get_image()
                            save_image(img, position, exposure,power*1000)
                            #time.sleep(0.05)
                            # print('pic num. : ', k, 'for position :', position)~
                time.sleep(10)
            except:
                print('An exception occured')
        if pulsed:
                laser.set2(0)
        else:
           laser.set(0)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Scanning for CW operation
Measurement for power:  6.75
204500
Data Set # 1
Now at Position :  10.0
An exception occured


In [ ]:
#POWER SCANS INCLUDED - Manually setting the exposition params.
%load_ext autoreload
%autoreload 2

import numpy as np
import Components as comp
import time
import socket
import sys
import os

if socket.gethostname() == "ph-photonbec5":
    sys.path.append("D:/Control/PythonPackages/")
    sys.path.append("D:/Control/PiezoController/")

    
from pbec_analysis import make_timestamp,ExperimentalDataSet, CameraData

def save_image(image,stage_position,exposure_time,pwr):
    timestamp = make_timestamp(precision=0)
    dataset = ExperimentalDataSet(timestamp)
    camera_data = CameraData(timestamp,extension='_' + str(round(stage_position,1)) + '_' + str(pwr) + '_' + str(int(exposure_time)) + '_' + str(np.amax(image)) + '_single' + '.png')
    camera_data.data = image
    dataset.dataset["CavityCamera" + str(stage_position) + str(pwr) + str(exposure_time) + str(np.amax(image))+ 'single'] = camera_data
    dataset.saveAllData()
    time.sleep(0.2)

min_exp = 100
max_exp = 1999994
number_of_pics = 10 # Amount of pics to take at each position of the stage
n_run = 1 # Amount of run to perform for each power
#_________________________________________________________CHANGE HERE____________________________________________________________
position_list = np.append(np.append(np.arange(0,90,5),np.arange(90,300,10)),np.arange(300,630,30))
position_list = np.arange(10,25,0.5)
power_list = np.append(np.append(np.arange(35,70,1)/1000,np.arange(70,100,5)/1000),np.arange(100,150,10)/1000) #CHANGE HERE
power = np.array([6.750])/1000
pulsed = False
exposure = 204500

filter_positions = { # Preset the filter position for each power value - Need to be done manually, do it everytime before taking measurements
    30: 1,
    35: 2,
    40: 3,
    50: 4,
    # Add more power settings as needed
}
#_________________________________________________________________________________________________________________________________

with comp.Thor_Camera(min_exp, max_exp, measure=True, max_frames=1) as camera:
    with comp.Translation_Stage() as stage:
        for j in range(n_run):
            
            print('Data Set #',j+1)

            for k in range(len(position_list)):
                stage.set(position_list[k])
                position = stage.get_position()
                print('Now at Position : ', position)

                for k in range(number_of_pics):
                    img = camera.get_image()
                    save_image(img, position, exposure,power*1000)
                    
                    #time.sleep(0.05)
                    # print('pic num. : ', k, 'for position :', position)~
            time.sleep(10)
        
        # for power in power_list:
        #     if pulsed:
        #         laser.set2(power)
        #         print('Scanning for pulsed operation')
        #     else:
        #         laser.set(power)
        #         print('Scanning for CW operation')
        #     # if power <=28/1000 and power >=27/1000:
        #     #     camera.change_exposure(1100)
        #     # else:
        #     #     camera.change_exposure(min_exp)
        #     print('Exposure is : ', camera.get_exposure())
        #     print('Measurement for power: ', power*1000)
        #     try:
        #         if power*1000 in filter_positions:
        #             filter_position = filter_positions[power*1000]
        #             filter_wheel.set(filter_position)  # Set the filter position accordingly
        #             print(f'Filter position set to: {filter_position+1} for power {power*1000}')
        #             
        #         for j in range(n_run):
        #             exposure = camera.get_exposure()
        #             print('Data Set #',j+1)
        # 
        #             for k in range(len(position_list)):
        #                 stage.set(position_list[k])
        #                 position = stage.get_position()
        #                 print('Now at Position : ', position)
        # 
        #                 for k in range(number_of_pics):
        #                     img = camera.get_image()
        #                     save_image(img, position, exposure,power*1000)
        #                     #time.sleep(0.05)
        #                     # print('pic num. : ', k, 'for position :', position)~
        #         time.sleep(10)
        #     except:
        #         print('An exception occured')
        # if pulsed:
        #         laser.set2(0)
        # else:

C:\Users\photonbec\AppData\Local\Programs\Python\Python38-32\lib\site-packages\pylablib\devices\Thorlabs\kinesis.py:230: UserWarning: model number BBD201 doesn't match the device ID prefix 73(BBD103)
  warnings.warn("model number {} doesn't match the device ID prefix {}({})".format(model_no,port,port_model_no))


Data Set # 1
Now at Position :  10.0
Now at Position :  10.5
